In [2]:
!pip install faiss-cpu numpy opencv-python insightface onnxruntime-gpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 8.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 28.4 MB/s eta 0:00:00
  Created wheel for insightface: filename=insightface-0.7.3-cp312-cp312-linux_x86_64.whl size=1071486 sha256=1f572d5c5288960b91b61f03f71c0621719fc05a2dfc7fc9f1e59a15d2bb92cd
  Stored in directory: /root/.cache/pip/wheels/73/3c/e2/6d4815e8a8b33a2006554d65ce0d1f973e768f4c7a222fa675
Successfully built insightface


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1장 인식

In [1]:
import os
import sys
import json
import numpy as np
import faiss

THRESHHOLD = 0.26

# 1. 작업 디렉토리 및 face_embedder 설정

WORKING_DIR = '/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플'

if WORKING_DIR not in sys.path:
    sys.path.insert(0, WORKING_DIR)

from face_embedder import FaceEmbedder

# 2. 경로 설정
INDEX_PATH = os.path.join(WORKING_DIR, 'face_database.index')
META_PATH = os.path.join(WORKING_DIR, 'face_metadata.json')
WEIGHT_PATH = os.path.join(WORKING_DIR, 'arc_r18_fp16_backbone.pth')

# 3. 데이터베이스 로드
print("⏳ FAISS 데이터베이스 및 메타데이터 로딩 중...")
index = faiss.read_index(INDEX_PATH)

with open(META_PATH, 'r', encoding='utf-8') as f:
    # JSON 키는 문자열로 저장되므로, 조회 편의를 위해 정수형(Int) 키로 변환합니다.
    id_to_metadata = {int(k): v for k, v in json.load(f).items()}
print("✅ DB 로드 완료!")

# 4. 임베딩 추출기 로드
embedder = FaceEmbedder(weight_path=WEIGHT_PATH)

# ----------------------------------------------------
# 🔍 실제 검색 함수 정의
# ----------------------------------------------------
def identify_face(image_path, threshold=THRESHHOLD):
    """
    쿼리 이미지의 신원을 파악합니다.
    """
    if not os.path.exists(image_path):
        print(f"❌ 이미지를 찾을 수 없습니다: {image_path}")
        return None

    # 임베딩 추출 및 float32 캐스팅
    query_emb = embedder.extract_batch([image_path], batch_size=1).astype('float32')

    if query_emb.shape[0] == 0:
        print("❌ 이미지에서 얼굴 임베딩을 추출하지 못했습니다.")
        return None

    # ⚠️ 중요: 계산 오차 방지를 위해 쿼리 벡터도 정규화 수행
    faiss.normalize_L2(query_emb)

    # FAISS 검색 (k=1: 가장 닮은 1명 찾기)
    # D: 유사도 점수(코사인 유사도), I: 매칭된 DB 내부 ID
    D, I = index.search(query_emb, k=1)

    matched_id = I[0][0]
    similarity = D[0][0]

    # 결과 판정
    if matched_id != -1 and similarity >= threshold:
        person_info = id_to_metadata[matched_id]
        print(f"✨ [식별 성공] {person_info['name']} (그룹: {person_info['group']})")
        print(f"   📊 유사도: {similarity:.4f} (기준치: {threshold})")
        return person_info, similarity
    else:
        print(f"⚠️ [식별 실패] 신원 미상 또는 일치하는 정보 없음 (최고 유사도: {similarity:.4f} / 기준치: {threshold})")
        return None, similarity

# --- 실행 예시 ---
# 테스트할 쿼리 이미지 경로를 넣고 함수를 실행해봅니다.
query_img_path = "/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/한국 연예인_전처리/한국연예인_AttackSet/류진/류진_정면_13_face_0.jpg"
result, score = identify_face(query_img_path, threshold=0.20)

⏳ FAISS 데이터베이스 및 메타데이터 로딩 중...
✅ DB 로드 완료!


/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/insightface/recognition/arcface_torch/backbones/iresnet.py:149: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self.fp16):


✨ [식별 성공] 류진 (그룹: korean)
   📊 유사도: 0.7313 (기준치: 0.26)


# 여러 장 인식 - 검증 안됨. 새 데이터셋 오면 할 예정.

In [5]:
import os
import sys
import json
import numpy as np
import faiss

# ----------------------------------------------------
# 1. 기본 설정 및 경로
# ----------------------------------------------------
WORKING_DIR = '/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플'
if WORKING_DIR not in sys.path:
    sys.path.insert(0, WORKING_DIR)

from face_embedder import FaceEmbedder

INDEX_PATH = os.path.join(WORKING_DIR, 'face_database.index')
META_PATH = os.path.join(WORKING_DIR, 'face_metadata.json')
WEIGHT_PATH = os.path.join(WORKING_DIR, 'arc_r18_fp16_backbone.pth')
ATTACK_DIR = '/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/한국 연예인_전처리/한국연예인_Attack'

# ----------------------------------------------------
# 2. 로드 파트
# ----------------------------------------------------
print("⏳ FAISS 데이터베이스 및 메타데이터 로딩 중...")
index = faiss.read_index(INDEX_PATH)

with open(META_PATH, 'r', encoding='utf-8') as f:
    id_to_metadata = {int(k): v for k, v in json.load(f).items()}

print("⏳ 임베딩 모델 로딩 중...")
embedder = FaceEmbedder(weight_path=WEIGHT_PATH)
print("✅ DB 및 모델 준비 완료!\n")

# ----------------------------------------------------
# 3. 데이터 수집 및 Rank-1 평가 함수
# ----------------------------------------------------
def evaluate_rank1_accuracy(attack_base_dir):
    image_paths = []
    true_labels = []

    print(f"📂 폴더 스캔 중: {attack_base_dir}")
    for root, dirs, files in os.walk(attack_base_dir):
        person_name = os.path.basename(root)
        for file in files:
            if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                image_paths.append(os.path.join(root, file))
                true_labels.append(person_name)

    total_images = len(image_paths)
    if total_images == 0:
        print("❌ 테스트할 이미지가 없습니다.")
        return

    print(f"⚡ 총 {total_images}장의 이미지 임베딩 추출 중...")
    embs = embedder.extract_batch(image_paths, batch_size=64).astype('float32')

    # 정규화 및 1:N 검색 (Rank-1을 위해 k=1로 설정)
    faiss.normalize_L2(embs)
    print("⚡ FAISS 1:N 검색 진행 중...")
    D, I = index.search(embs, k=1)

    # ------------------------------------------------
    # 🥇 Rank-1 Accuracy 채점
    # ------------------------------------------------
    rank1_correct = 0

    for i in range(total_images):
        matched_id = I[i][0]
        actual_name = true_labels[i]

        if matched_id != -1:
            predicted_name = id_to_metadata[matched_id]['name']

            # 임계값 필터 없이, 찾아온 1등의 이름이 정답과 일치하는지만 대조
            if predicted_name == actual_name:
                rank1_correct += 1

    rank1_acc = (rank1_correct / total_images) * 100

    # ------------------------------------------------
    # 📢 최종 리포트 출력
    # ------------------------------------------------
    print("\n" + "="*55)
    print(" 🏆 [1:N 얼굴 식별 시스템 성능 평가]")
    print("="*55)
    print(f" 📌 총 테스트 이미지 : {total_images} 장")
    print(f" 🎯 정답 매칭 성공   : {rank1_correct} 장")
    print(f" 🚨 오식별 및 실패   : {total_images - rank1_correct} 장")
    print("-" * 55)
    print(f" 🥇 Rank-1 Accuracy  : {rank1_acc:.2f}%")
    print("    (임계값 제한 없이, 시스템이 1등으로 지목한 신원이 진짜 본인일 확률)")
    print("="*55)

# ----------------------------------------------------
# 4. 실행
# ----------------------------------------------------
if __name__ == "__main__":
    evaluate_rank1_accuracy(ATTACK_DIR)

⏳ FAISS 데이터베이스 및 메타데이터 로딩 중...
⏳ 임베딩 모델 로딩 중...
✅ DB 및 모델 준비 완료!

📂 폴더 스캔 중: /content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/한국 연예인_전처리/한국연예인_Attack
⚡ 총 307장의 이미지 임베딩 추출 중...


/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/insightface/recognition/arcface_torch/backbones/iresnet.py:149: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self.fp16):


⚡ FAISS 1:N 검색 진행 중...

 🏆 [1:N 얼굴 식별 시스템 성능 평가]
 📌 총 테스트 이미지 : 307 장
 🎯 정답 매칭 성공   : 303 장
 🚨 오식별 및 실패   : 4 장
-------------------------------------------------------
 🥇 Rank-1 Accuracy  : 98.70%
    (임계값 제한 없이, 시스템이 1등으로 지목한 신원이 진짜 본인일 확률)
